In [88]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                           ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
    
from sklearn.preprocessing import PowerTransformer


In [89]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [90]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [91]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [92]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [93]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [94]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [95]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [96]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [97]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [98]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [99]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [100]:
# Set lower, upper time point and times for IBS calculation later 

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 14)
y_train:  (139,)


(99, 16)

# Feature Selection: PLSR

In [101]:
selected_features = [
"MTV",
"TLG",
"hpv_related",
"SUVpeak",
"uicc8_III-IV",
"oropharynx",
"cavum_oris",
"charlson",
"hypopharynx"
]

# Selecting features in the DataFrame
X_plsr = X[selected_features]
X_new = X_plsr.copy()

In [102]:
X_MAASTRO_plsr = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_plsr.copy()

# Yeo-Johnson Transformation

In [103]:
# Transform X_new 
# Set the categorical_columns
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Apply Yeo-Johnson transformation
pt = PowerTransformer(method='yeo-johnson')
X_new_numeric_transformed = pt.fit_transform(X_new_numeric)

# Create DataFrame with transformed numerical data
X_new_numeric_transformed = pd.DataFrame(X_new_numeric_transformed, 
                                         columns=X_new_numeric.columns, 
                                         index=X_new.index)

# Concatenate transformed numerical data with categorical data
X_new_std = pd.concat([X_new_numeric_transformed, X_new_categoric], axis=1)
X_new_std = X_new_std[X_new.columns]


# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the transformation for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = pt.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [104]:
X_new

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx,cavum_oris,charlson,hypopharynx
0,7.934,86.228420,0.0,14.473272,0.0,1,0,0,0
1,1.656,7.040100,0.0,5.044678,0.0,0,0,1,0
2,14.502,83.569669,0.0,7.839043,1.0,0,1,1,0
3,2.440,5.567091,0.0,2.880631,0.0,0,0,1,0
4,3.668,16.150550,0.0,5.402006,0.0,0,0,1,0
...,...,...,...,...,...,...,...,...,...
134,3.650,26.280140,1.0,9.290139,0.0,1,0,0,0
135,18.967,101.754834,1.0,7.172883,1.0,1,0,0,0
136,6.370,66.273201,1.0,13.873187,0.0,1,0,1,0
137,12.443,71.832443,1.0,7.507419,1.0,1,0,1,0


In [105]:
X_new_std

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx,cavum_oris,charlson,hypopharynx
0,0.080033,0.343545,0.0,0.750782,0.0,1,0,0,0
1,-1.722060,-1.657833,0.0,-1.213660,0.0,0,0,1,0
2,0.735716,0.318631,0.0,-0.474112,1.0,0,1,1,0
3,-1.291020,-1.835650,0.0,-1.992620,0.0,0,0,1,0
4,-0.816679,-1.003396,0.0,-1.106304,0.0,0,0,1,0
...,...,...,...,...,...,...,...,...,...
134,-0.822461,-0.611074,1.0,-0.158214,0.0,1,0,0,0
135,1.008070,0.474941,1.0,-0.632361,1.0,1,0,0,0
136,-0.170908,0.133612,1.0,0.658564,0.0,1,0,1,0
137,0.574668,0.197985,1.0,-0.551725,1.0,1,0,1,0


In [106]:
MAASTRO_new

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx,cavum_oris,charlson,hypopharynx
0,22.841,263.611623,1,15.438583,0,1,0,1,0
1,5.660,36.980700,0,8.829353,1,1,0,0,0
2,7.791,74.636342,0,13.476123,1,1,0,1,0
3,7.908,46.791979,0,8.632732,1,0,0,1,0
4,15.237,107.637514,1,9.783954,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...
94,6.110,144.490782,0,31.338410,1,0,0,0,0
95,7.182,69.214868,0,13.041604,1,0,0,1,0
96,16.483,102.594274,1,8.944517,1,1,0,1,0
97,9.981,103.229492,1,14.184236,0,1,0,0,0


In [107]:
MAASTRO_new_std

,MTV,TLG,hpv_related,SUVpeak,uicc8_III-IV,oropharynx,cavum_oris,charlson,hypopharynx
0,1.188930,1.218887,1,0.893634,0,1,0,1,0
1,-0.307894,-0.335276,0,-0.254678,1,1,0,0,0
2,0.059450,0.228548,0,0.595996,1,1,0,1,0
3,0.076321,-0.145668,0,-0.296877,1,0,0,1,0
4,0.786849,0.519421,1,-0.058378,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...
94,-0.219083,0.751363,0,2.645651,1,0,0,0,0
95,-0.033134,0.168333,0,0.526028,1,0,0,1,0
96,0.867254,0.481447,1,-0.230255,1,1,0,1,0
97,0.336049,0.486335,1,0.706707,0,1,0,0,0


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [108]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:58:50,423] A new study created in memory with name: no-name-406643e6-b2f0-4d7b-b4d4-eaf77c471f39


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.8059071729957806


[I 2024-04-13 23:58:52,512] A new study created in memory with name: no-name-e66d87fe-fbec-4009-8c80-aba27bfc2deb


Fold 5 C-index: 0.6338028169014085
[I 2024-04-13 23:58:52,483] Trial 0 finished with value: 0.7373181120614343 and parameters: {}. Best is trial 0 with value: 0.7373181120614343.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7373181120614343], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 50, 778624), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 52, 482453), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7373181120614343


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16450084983071073
Fold 2 IBS: 0.19911115381589756
Fold 3 IBS: 0.16198548420238557
Fold 4 IBS: 0.1434899954309807
Fold 5 IBS: 0.2643634750339842
[I 2024-04-13 23:58:55,778] Trial 0 finished with value: 0.18669019166279174 and parameters: {}. Best is trial 0 with value: 0.18669019166279174.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.18669019166279174], datetime_start=datetime.datetime(2024, 4, 13, 23, 58, 52, 559953), datetime_complete=datetime.datetime(2024, 4, 13, 23, 58, 55, 756198), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.18669019166279174


In [109]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [110]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.737
train_ibs:  0.187


#### Test

In [111]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [112]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.593
IBS score: 0.269


In [113]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [114]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [115]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:59:01,177] A new study created in memory with name: no-name-d93d378f-1e27-409b-bb7b-95385df4d183


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.5411255411255411
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.6985294117647058
Fold 4 C-index: 0.7215189873417721


[I 2024-04-13 23:59:03,258] A new study created in memory with name: no-name-9568f296-570c-4f21-be57-60c2c682992a


Fold 5 C-index: 0.6338028169014085
[I 2024-04-13 23:59:03,144] Trial 0 finished with value: 0.6466739228552569 and parameters: {}. Best is trial 0 with value: 0.6466739228552569.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6466739228552569], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 1, 410427), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 3, 144395), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6466739228552569


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397652106674792
Fold 2 IBS: 0.2215779146949956
Fold 3 IBS: 0.20453593853798394
Fold 4 IBS: 0.22473802939707874
Fold 5 IBS: 0.218124313086617
[I 2024-04-13 23:59:05,470] Trial 0 finished with value: 0.21659054335668465 and parameters: {}. Best is trial 0 with value: 0.21659054335668465.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054335668465], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 3, 449768), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 5, 467549), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054335668465


In [116]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [117]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.647
train_ibs:  0.217


#### Test

In [118]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [119]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.701


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [120]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [121]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:59:06,801] A new study created in memory with name: no-name-09b65462-2373-4275-a365-d71eb0b9b730


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7932489451476793


[I 2024-04-13 23:59:09,713] A new study created in memory with name: no-name-d01dd162-319a-44e5-a927-9727f0ebe9cd


Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:59:09,687] Trial 0 finished with value: 0.7356108423369023 and parameters: {}. Best is trial 0 with value: 0.7356108423369023.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7356108423369023], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 6, 831896), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 9, 684013), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7356108423369023


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16493530008837418
Fold 2 IBS: 0.20469114844351927
Fold 3 IBS: 0.1603629735865767
Fold 4 IBS: 0.14338296717670074
Fold 5 IBS: 0.26373792195731416
[I 2024-04-13 23:59:14,937] Trial 0 finished with value: 0.187422062250497 and parameters: {}. Best is trial 0 with value: 0.187422062250497.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.187422062250497], datetime_start=datetime.datetime(2024, 4, 13, 23, 59, 10, 256861), datetime_complete=datetime.datetime(2024, 4, 13, 23, 59, 14, 936686), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.187422062250497


In [122]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [123]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.187


#### Test 

In [124]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [125]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.597


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.272


In [126]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [127]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:59:17,064] A new study created in memory with name: no-name-b8c05738-c837-4e94-8a09-79b0a1fbbba1


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:59:20,083] Trial 0 finished with value: 0.7357254336279643 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7357254336279643.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:59:22,263] Trial 1 finished with value: 0.7347450414711015 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7357254336279643.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:59:25,555] Trial 2 finished with value: 0.7347450414711015 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.73572543362

Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:00:54,809] Trial 25 finished with value: 0.734786466491814 and parameters: {'l1_ratio': 0.535645013796375}. Best is trial 0 with value: 0.7357254336279643.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 00:00:57,746] Trial 26 finished with value: 0.7347450414711015 and parameters: {'l1_ratio': 0.3312072877416866}. Best is trial 0 with value: 0.7357254336279643.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:01:00,880] Trial 27 finished with value: 0.7338060743349513 and parameters: {'l1_ratio': 0.6634231383978803}. Best is trial 0 with value: 0.735725433627

Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:02:01,658] Trial 50 finished with value: 0.7338060743349513 and parameters: {'l1_ratio': 0.41653985858071785}. Best is trial 0 with value: 0.7357254336279643.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7647058823529411
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:02:04,523] Trial 51 finished with value: 0.7338060743349513 and parameters: {'l1_ratio': 0.6094269520256451}. Best is trial 0 with value: 0.7357254336279643.
Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.7932489451476793
Fold 5 C-index: 0.6338028169014085
[I 2024-04-14 00:02:07,480] Trial 52 finished with value: 0.734786466491814 and parameters: {'l1_ratio': 0.5861258204213798}. Best is trial 0 with value: 0.7357254336

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.16469562500727153
Fold 2 IBS: 0.20449731917283143
Fold 3 IBS: 0.1600719084718141
Fold 4 IBS: 0.14343027006612022
Fold 5 IBS: 0.2636538440298299
[I 2024-04-14 00:04:04,276] Trial 0 finished with value: 0.18726979334957342 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.18726979334957342.
Fold 1 IBS: 0.1647003456291085
Fold 2 IBS: 0.204041943438731
Fold 3 IBS: 0.16135734558481252
Fold 4 IBS: 0.14346263739572104
Fold 5 IBS: 0.2636476971898714
[I 2024-04-14 00:04:06,670] Trial 1 finished with value: 0.1874419938476489 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.18726979334957342.
Fold 1 IBS: 0.1646944271302574
Fold 2 IBS: 0.20394851497628436
Fold 3 IBS: 0.16131777936473066
Fold 4 IBS: 0.14349730341343223
Fold 5 IBS: 0.2636546829900082
[I 2024-04-14 00:04:08,854] Trial 2 finished with value: 0.18742254157494256 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.18726979334957342.

Fold 1 IBS: 0.16492302770825934
Fold 2 IBS: 0.20459171325445838
Fold 3 IBS: 0.16029028566299094
Fold 4 IBS: 0.14345934583032743
Fold 5 IBS: 0.2636822114088275
[I 2024-04-14 00:05:00,770] Trial 25 finished with value: 0.1873893167729727 and parameters: {'l1_ratio': 0.8277117787416304}. Best is trial 8 with value: 0.18720119127870555.
Fold 1 IBS: 0.16464502137730158
Fold 2 IBS: 0.20439349740742072
Fold 3 IBS: 0.16144555475428507
Fold 4 IBS: 0.14348015102692974
Fold 5 IBS: 0.2636850530467372
[I 2024-04-14 00:05:02,866] Trial 26 finished with value: 0.18752985552253482 and parameters: {'l1_ratio': 0.5711012798301316}. Best is trial 8 with value: 0.18720119127870555.
Fold 1 IBS: 0.1648742737962242
Fold 2 IBS: 0.20456045631195469
Fold 3 IBS: 0.16027496193957216
Fold 4 IBS: 0.14338878396514534
Fold 5 IBS: 0.2636656853757507
[I 2024-04-14 00:05:04,808] Trial 27 finished with value: 0.18735283227772942 and parameters: {'l1_ratio': 0.7581476603189202}. Best is trial 8 with value: 0.1872011912787

Fold 1 IBS: 0.1646233455981672
Fold 2 IBS: 0.204471613951031
Fold 3 IBS: 0.1601195550835879
Fold 4 IBS: 0.14348509667238624
Fold 5 IBS: 0.26366169510967513
[I 2024-04-14 00:06:00,291] Trial 50 finished with value: 0.1872722612829695 and parameters: {'l1_ratio': 0.6531713347516178}. Best is trial 8 with value: 0.18720119127870555.
Fold 1 IBS: 0.1646569893970897
Fold 2 IBS: 0.2043609120872407
Fold 3 IBS: 0.16003633723065996
Fold 4 IBS: 0.1434352670057949
Fold 5 IBS: 0.26369343709403986
[I 2024-04-14 00:06:02,361] Trial 51 finished with value: 0.18723658856296505 and parameters: {'l1_ratio': 0.5344975351962169}. Best is trial 8 with value: 0.18720119127870555.
Fold 1 IBS: 0.16464257898199788
Fold 2 IBS: 0.20430705427785487
Fold 3 IBS: 0.16144530551933242
Fold 4 IBS: 0.14346883478293687
Fold 5 IBS: 0.2636872612956089
[I 2024-04-14 00:06:04,584] Trial 52 finished with value: 0.1875102069715462 and parameters: {'l1_ratio': 0.4915091696239893}. Best is trial 8 with value: 0.18720119127870555.

Fold 1 IBS: 0.16463799926571732
Fold 2 IBS: 0.20431965754133014
Fold 3 IBS: 0.16141304465920647
Fold 4 IBS: 0.14349013355707313
Fold 5 IBS: 0.26364421037649893
[I 2024-04-14 00:06:59,137] Trial 75 finished with value: 0.1875010090799652 and parameters: {'l1_ratio': 0.5041192360230345}. Best is trial 8 with value: 0.18720119127870555.
Fold 1 IBS: 0.16464859813249
Fold 2 IBS: 0.20438415691590445
Fold 3 IBS: 0.16010991283393333
Fold 4 IBS: 0.14347351757544885
Fold 5 IBS: 0.26366815606211175
[I 2024-04-14 00:07:01,411] Trial 76 finished with value: 0.18725686830397764 and parameters: {'l1_ratio': 0.5595001145684224}. Best is trial 8 with value: 0.18720119127870555.
Fold 1 IBS: 0.1646079927827805
Fold 2 IBS: 0.20454688189514722
Fold 3 IBS: 0.16018894183323895
Fold 4 IBS: 0.1434701781534836
Fold 5 IBS: 0.2636968931690041
[I 2024-04-14 00:07:03,414] Trial 77 finished with value: 0.18730217756673087 and parameters: {'l1_ratio': 0.7318055230343666}. Best is trial 8 with value: 0.187201191278705

In [128]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [129]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.187


#### Test

In [130]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [131]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.598


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.47850235241861494)

test_ibs:  0.272


In [132]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [163]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 12:48:37,261] A new study created in memory with name: no-name-eaa50ade-873f-422a-9a9f-e609614f02a3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.6830357142857143
Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.636150234741784
[I 2024-04-14 12:48:40,495] Trial 0 finished with value: 0.7405641982153078 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7405641982153078.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.676056338028169
[I 2024-04-14 12:48:43,049] Trial 1 finished with value: 0.7382735828654546 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_f

Fold 1 C-index: 0.6601731601731602
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.7230392156862745
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 12:49:19,369] Trial 16 finished with value: 0.7452471942033755 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 1, 'n_estimators': 12, 'oob_score': True, 'max_samples': 0.4475428611563601, 'max_features': None, 'min_weight_fraction_leaf': 0.0888512086404308, 'warm_start': True}. Best is trial 15 with value: 0.7652207475317394.
Fold 1 C-index: 0.7056277056277056
Fold 2 C-index: 0.8058035714285714
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.7230046948356808
[I 2024-04-14 12:49:20,728] Trial 17 finished with value: 0.7732336823406651 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 16, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 119, 'oob_score': True, 'max_samples': 0.8531582264400189, 'm

Fold 1 C-index: 0.6731601731601732
Fold 2 C-index: 0.8258928571428571
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8649789029535865
Fold 5 C-index: 0.7230046948356808
[I 2024-04-14 12:49:38,943] Trial 31 finished with value: 0.7772112471870869 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 17, 'min_samples_leaf': 17, 'max_depth': 7, 'n_estimators': 166, 'oob_score': True, 'max_samples': 0.9924754281584311, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.14400488960666644, 'warm_start': True}. Best is trial 25 with value: 0.7848431571854013.
Fold 1 C-index: 0.6883116883116883
Fold 2 C-index: 0.7879464285714286
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.869198312236287
Fold 5 C-index: 0.7065727699530516
[I 2024-04-14 12:49:39,741] Trial 32 finished with value: 0.7751117221674322 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 17, 'min_samples_leaf': 19, 'max_depth': 8, 'n_estimators': 132, 'oob_score': True, 'max_samples': 0.9291992556079912

...

Fold 5 IBS: 0.22005380670626368
[I 2024-04-14 12:54:04,184] Trial 31 finished with value: 0.1904280642507993 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 17, 'min_samples_leaf': 2, 'max_depth': 15, 'n_estimators': 294, 'oob_score': False, 'max_samples': 0.9801990513520231, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3287325033473082}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.1798801675037484
Fold 2 IBS: 0.20212655757633474
Fold 3 IBS: 0.17228268183458786
Fold 4 IBS: 0.17709018730697135
Fold 5 IBS: 0.2171759766259116
[I 2024-04-14 12:54:06,750] Trial 32 finished with value: 0.18971111416951078 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 16, 'n_estimators': 270, 'oob_score': False, 'max_samples': 0.9502203520505428, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.3560337622605024}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.19901138494030404
Fold 2 IBS: 0.20449

Fold 1 IBS: 0.18550379267172096
Fold 2 IBS: 0.21942995999964232
Fold 3 IBS: 0.16553621749747308
Fold 4 IBS: 0.15227009177089657
Fold 5 IBS: 0.22810092243528043
[I 2024-04-14 12:54:50,482] Trial 47 finished with value: 0.1901681968750027 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 9, 'min_samples_leaf': 6, 'max_depth': 5, 'n_estimators': 418, 'oob_score': True, 'max_samples': 0.5827826014419875, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.06743707678818638}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.1890195171243487
Fold 2 IBS: 0.2115470997677741
Fold 3 IBS: 0.16677823033546224
Fold 4 IBS: 0.16290595324316187
Fold 5 IBS: 0.22288495726733487
[I 2024-04-14 12:54:54,849] Trial 48 finished with value: 0.19062715154761634 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 2, 'n_estimators': 458, 'oob_score': False, 'max_samples': 0.3891906948262243, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 5 IBS: 0.22281161806162805
[I 2024-04-14 12:55:32,163] Trial 62 finished with value: 0.18834166758515986 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 16, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 233, 'oob_score': False, 'max_samples': 0.5443419407354706, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.05508766374457823}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.18363533539050453
Fold 2 IBS: 0.21416998096689313
Fold 3 IBS: 0.164866110696086
Fold 4 IBS: 0.15364072515883115
Fold 5 IBS: 0.22209693010700793
[I 2024-04-14 12:55:34,647] Trial 63 finished with value: 0.18768181646386456 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 2, 'n_estimators': 204, 'oob_score': False, 'max_samples': 0.6213136733560154, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.05547383035537368}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.1860680402386841
Fold 2 IBS: 0.21636

Fold 1 IBS: 0.18755610055924535
Fold 2 IBS: 0.213667461857247
Fold 3 IBS: 0.16673545659559605
Fold 4 IBS: 0.16836861050203789
Fold 5 IBS: 0.22211605013296412
[I 2024-04-14 12:55:56,527] Trial 78 finished with value: 0.1916887359294181 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 13, 'min_samples_leaf': 3, 'max_depth': 3, 'n_estimators': 204, 'oob_score': False, 'max_samples': 0.7004531022711529, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.1840602311535524}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.18688920841087794
Fold 2 IBS: 0.21267348472465292
Fold 3 IBS: 0.16215180077755184
Fold 4 IBS: 0.15379535963668461
Fold 5 IBS: 0.22306656476174708
[I 2024-04-14 12:55:57,659] Trial 79 finished with value: 0.1877152836623029 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 14, 'min_samples_leaf': 3, 'max_depth': 2, 'n_estimators': 105, 'oob_score': False, 'max_samples': 0.5637611791475481, 'max_features': 'auto', 'min_weight_fraction_leaf

Fold 5 IBS: 0.23505030209245742
[I 2024-04-14 12:56:15,625] Trial 93 finished with value: 0.19003337673849088 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 3, 'n_estimators': 67, 'oob_score': False, 'max_samples': 0.6206641744799919, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.033188309865125526}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.18730646376000343
Fold 2 IBS: 0.20758047942586721
Fold 3 IBS: 0.1722304326118517
Fold 4 IBS: 0.17236254605622933
Fold 5 IBS: 0.22175559456656074
[I 2024-04-14 12:56:16,167] Trial 94 finished with value: 0.1922471032841025 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 16, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 38, 'oob_score': False, 'max_samples': 0.5052122145733884, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.009395619419171504}. Best is trial 11 with value: 0.18498244011454343.
Fold 1 IBS: 0.18680985415580556
Fold 2 IBS: 0.2200

In [164]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [165]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.787
train_ibs:  0.185


#### Test

In [166]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [167]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=12, max_features='auto', max_leaf_nodes=11,
                     max_samples=0.9415203168151968, min_samples_leaf=19,
                     min_samples_split=17,
                     min_weight_fraction_leaf=0.30712180280602136,
                     n_estimators=274, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.708


RandomSurvivalForest(max_depth=3, max_features='log2', max_leaf_nodes=18,
                     max_samples=0.9878564362760905, min_samples_leaf=1,
                     min_samples_split=2,
                     min_weight_fraction_leaf=0.008231293935378081,
                     n_estimators=287, random_state=123)

test_ibs:  0.208


In [168]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [169]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 12:56:23,457] A new study created in memory with name: no-name-3918d267-e299-41fc-ae19-95ba46855fec


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.8578431372549019
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 12:56:24,427] Trial 0 finished with value: 0.7787743763285959 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7787743763285959.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:56:26,564] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}. Best is t

Fold 1 C-index: 0.7597402597402597
Fold 2 C-index: 0.8147321428571429
Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 12:56:52,957] Trial 16 finished with value: 0.7756419137412984 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7814432639860918.
Fold 1 C-index: 0.7554112554112554
Fold 2 C-index: 0.7924107142857143
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8037974683544303
Fold 5 C-index: 0.6384976525821596
[I 2024-04-14 12:56:53,840] Trial 17 finished with value: 0.7646900847933786 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 318, 'oob_score': False, 'warm_start': True, 'max_feat

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:57:11,438] Trial 31 finished with value: 0.5 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 408, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.72769080216046, 'min_weight_fraction_leaf': 0.41272796590787575}. Best is trial 12 with value: 0.7814432639860918.
Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7924107142857143
Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8248945147679325
Fold 5 C-index: 0.6455399061032864
[I 2024-04-14 12:57:12,176] Trial 32 finished with value: 0.7708654365042669 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 4, 'max_depth': 16, 'n_estimators': 289, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8366845330876531, 'min_weight_fraction_leaf': 0.3478490479879

...

Fold 1 IBS: 0.17098851044726665
Fold 2 IBS: 0.21481315097086512
Fold 3 IBS: 0.16632603025301468
Fold 4 IBS: 0.16401546065620026
Fold 5 IBS: 0.2278634394044985
[I 2024-04-14 12:59:49,862] Trial 30 finished with value: 0.18880131834636904 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 6, 'n_estimators': 133, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.5318918739079888, 'min_weight_fraction_leaf': 0.10732787343717065}. Best is trial 18 with value: 0.18592544423505888.
Fold 1 IBS: 0.16709338870083962
Fold 2 IBS: 0.2207970817898967
Fold 3 IBS: 0.16106031320634173
Fold 4 IBS: 0.15411080176278658
Fold 5 IBS: 0.23492661118338568
[I 2024-04-14 12:59:53,926] Trial 31 finished with value: 0.18759763932865006 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 6, 'min_samples_leaf': 5, 'max_depth': 10, 'n_estimators': 458, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 

Fold 1 IBS: 0.16919093793006681
Fold 2 IBS: 0.22072139997365403
Fold 3 IBS: 0.16323839347363078
Fold 4 IBS: 0.15681709166454283
Fold 5 IBS: 0.23415423999459772
[I 2024-04-14 13:00:17,038] Trial 45 finished with value: 0.18882441260729843 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 4, 'min_samples_leaf': 8, 'max_depth': 14, 'n_estimators': 406, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.848529004532656, 'min_weight_fraction_leaf': 0.027746600647391314}. Best is trial 42 with value: 0.18536413936986168.
Fold 1 IBS: 0.17088136388053118
Fold 2 IBS: 0.22055326503970013
Fold 3 IBS: 0.16296852866957956
Fold 4 IBS: 0.15728088459397643
Fold 5 IBS: 0.23399061285989559
[I 2024-04-14 13:00:19,285] Trial 46 finished with value: 0.1891349310087366 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 5, 'min_samples_leaf': 6, 'max_depth': 12, 'n_estimators': 324, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples':

Fold 1 IBS: 0.18690585538594123
Fold 2 IBS: 0.20938648863281817
Fold 3 IBS: 0.181022371952325
Fold 4 IBS: 0.19009687205800668
Fold 5 IBS: 0.2170173183599625
[I 2024-04-14 13:01:01,746] Trial 60 finished with value: 0.1968857812778107 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 4, 'min_samples_leaf': 12, 'max_depth': 19, 'n_estimators': 352, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.8876535776296195, 'min_weight_fraction_leaf': 0.08357065616960663}. Best is trial 42 with value: 0.18536413936986168.
Fold 1 IBS: 0.16859725994611574
Fold 2 IBS: 0.21968660391194714
Fold 3 IBS: 0.16222130074755978
Fold 4 IBS: 0.15526729388868263
Fold 5 IBS: 0.2352496659850785
[I 2024-04-14 13:01:04,793] Trial 61 finished with value: 0.18820442489587677 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 5, 'min_samples_leaf': 7, 'max_depth': 17, 'n_estimators': 456, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.80287

Fold 1 IBS: 0.17119567250301165
Fold 2 IBS: 0.24105206157891446
Fold 3 IBS: 0.15213609072622497
Fold 4 IBS: 0.14859184553071816
Fold 5 IBS: 0.24326056635040982
[I 2024-04-14 13:01:46,676] Trial 75 finished with value: 0.1912472473378558 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 19, 'n_estimators': 453, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.6510484272191477, 'min_weight_fraction_leaf': 0.0708680095858405}. Best is trial 42 with value: 0.18536413936986168.
Fold 1 IBS: 0.18853691736578462
Fold 2 IBS: 0.209428188689307
Fold 3 IBS: 0.18272779365407454
Fold 4 IBS: 0.19346798348237323
Fold 5 IBS: 0.21682995331709484
[I 2024-04-14 13:01:50,968] Trial 76 finished with value: 0.19819816730172685 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 5, 'min_samples_leaf': 13, 'max_depth': 18, 'n_estimators': 491, 'oob_score': True, 'warm_start': False, 'max_features': 1, 'max_samples': 0.819792

Fold 1 IBS: 0.16995945205700103
Fold 2 IBS: 0.22143925404090192
Fold 3 IBS: 0.16372958458946052
Fold 4 IBS: 0.15806194585094185
Fold 5 IBS: 0.23432624599230448
[I 2024-04-14 13:02:31,277] Trial 90 finished with value: 0.18950329650612197 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 18, 'n_estimators': 468, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.9979423948457556, 'min_weight_fraction_leaf': 0.1267762525048648}. Best is trial 42 with value: 0.18536413936986168.
Fold 1 IBS: 0.16294257869738282
Fold 2 IBS: 0.22247116644247825
Fold 3 IBS: 0.1575456290734587
Fold 4 IBS: 0.150103345920691
Fold 5 IBS: 0.23188419808478244
[I 2024-04-14 13:02:34,490] Trial 91 finished with value: 0.18498938364375864 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 4, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 374, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.

In [170]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [171]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.798
train_ibs:  0.184


#### Test

In [172]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [173]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=20, max_features='auto', max_leaf_nodes=18,
                   max_samples=0.4107674575173371, min_samples_leaf=1,
                   min_samples_split=17,
                   min_weight_fraction_leaf=0.005571403264584145,
                   n_estimators=192, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.632


ExtraSurvivalTrees(max_depth=17, max_features='log2', max_leaf_nodes=6,
                   max_samples=0.8853584450011042, min_samples_leaf=2,
                   min_samples_split=15,
                   min_weight_fraction_leaf=0.00885577315322818,
                   n_estimators=322, random_state=123, warm_start=True)

IBS: 0.217


In [174]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [175]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 13:02:55,227] A new study created in memory with name: no-name-77de4da3-4372-48b6-a0a4-4c6faec7a37b


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 13:03:08,618] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 13:03:15,955] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:30:07,545] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7287367685773478.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:30:32,376] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:40:23,691] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7287367685773478.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5558035714285714
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6197183098591549
[I 2024-04-14 14:41:00,229] Trial 26 finished with value: 0.5351043762575453 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:27:43,181] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.21504986372331816.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:28:17,291] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.8324789538052517, 'learning_rate': 0.09850922090204048, 'dropout_rate': 0.2217299975485909, 'n_estimators': 39

Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:36:38,847] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8639813037885321, 'learning_rate': 0.010366561182703663, 'dropout_rate': 0.2054023171521986, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 1.0590752315578722, 'min_weight_fraction_leaf': 0.1974211678184809, 'max_features': 'log2', 'min_impurity_decrease': 4.836772048236878e-06, 'validation_fraction': 0.9971715713668047, 'min_samples_split': 18, 'max_leaf_nodes': 14, 'min_samples_leaf': 11, 'max_depth': 4}. Best is trial 53 with value: 0.21504986372331816.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609557
[I 2024-04-14 15:37:42,859] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.924711151270876, 'learning_rate': 0.009416377941785207, 'dropout_rate': 0.1662721971023693, 'n_estimators': 48

Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:44:56,957] Trial 77 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9392590368987991, 'learning_rate': 0.011132229721027662, 'dropout_rate': 0.25203573700169807, 'n_estimators': 441, 'criterion': 'squared_error', 'ccp_alpha': 0.33381590515318865, 'min_weight_fraction_leaf': 0.13051249394026804, 'max_features': 1, 'min_impurity_decrease': 7.70123021698274e-05, 'validation_fraction': 0.994934454010795, 'min_samples_split': 18, 'max_leaf_nodes': 13, 'min_samples_leaf': 18, 'max_depth': 3}. Best is trial 53 with value: 0.21504986372331816.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:45:46,395] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8479726466301248, 'learning_rate': 0.04205291686981777, 'dropout_rate': 0.1983089540

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609574
[I 2024-04-14 15:51:50,365] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8335693537780458, 'learning_rate': 0.025896998029808806, 'dropout_rate': 0.2592233887917012, 'n_estimators': 416, 'criterion': 'squared_error', 'ccp_alpha': 1.998777407817298, 'min_weight_fraction_leaf': 0.06882085091143297, 'max_features': 'log2', 'min_impurity_decrease': 2.1900682995121406e-06, 'validation_fraction': 0.9382598277453086, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 18, 'max_depth': 3}. Best is trial 85 with value: 0.21397187071021745.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:51:58,438] Trial 89 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.8801139289725153, 'learning_rate': 0.0203835198

Fold 3 IBS: 0.2031414835960969
Fold 4 IBS: 0.22300087224173537
Fold 5 IBS: 0.2178046738886567
[I 2024-04-14 15:57:27,774] Trial 99 finished with value: 0.21556453497499795 and parameters: {'subsample': 0.9762156523558686, 'learning_rate': 0.01377474153403631, 'dropout_rate': 0.2717132298988158, 'n_estimators': 463, 'criterion': 'squared_error', 'ccp_alpha': 0.004394120129034387, 'min_weight_fraction_leaf': 0.28881659395589254, 'max_features': 'auto', 'min_impurity_decrease': 1.3534409368328512e-06, 'validation_fraction': 0.6748895535913024, 'min_samples_split': 19, 'max_leaf_nodes': 12, 'min_samples_leaf': 13, 'max_depth': 17}. Best is trial 85 with value: 0.21397187071021745.


* Best trial for IBS: 
 FrozenTrial(number=85, state=TrialState.COMPLETE, values=[0.21397187071021745], datetime_start=datetime.datetime(2024, 4, 14, 15, 50, 36, 523157), datetime_complete=datetime.datetime(2024, 4, 14, 15, 50, 52, 617914), params={'subsample': 0.8559905830599279, 'learning_rate': 0.01455486647

In [176]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [177]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.743
train_ibs:  0.214


#### Test

In [178]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [179]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.59


GradientBoostingSurvivalAnalysis(ccp_alpha=0.011241474018604543,
                                 criterion='squared_error',
                                 dropout_rate=0.10391090561180716,
                                 learning_rate=0.014554866476952134,
                                 max_depth=1, max_features='auto',
                                 max_leaf_nodes=18,
                                 min_impurity_decrease=1.8697047323397577e-06,
                                 min_samples_leaf=19, min_samples_split=20,
                                 min_weight_fraction_leaf=0.12733224205170357,
                                 n_estimators=265, random_state=123,
                                 subsample=0.8559905830599279,
                                 validation_fraction=0.9999275626013308)

IBS: 0.218


In [180]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [181]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Transformation
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            pt = PowerTransformer(method='yeo-johnson')
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = pt.fit_transform(X_train_included)
                X_test_included_std = pt.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 15:57:35,270] A new study created in memory with name: no-name-4dcdc826-72cd-4f82-967b-69010ed42821


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 15:57:36,663] Trial 0 finished with value: 0.6849872959240584 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6849872959240584.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 15:57:44,545] Trial 1 finished with value: 0.6943010214142545 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 1 with value: 0.6943010214142545.
Fold 1 C-index: 0.7251082251082251
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784


Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7142857142857143
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.647887323943662
[I 2024-04-14 15:58:57,512] Trial 19 finished with value: 0.7378573705369926 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9820749239416067, 'n_estimators': 431, 'learning_rate': 0.08300323114610605}. Best is trial 19 with value: 0.7378573705369926.
Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 15:59:02,793] Trial 20 finished with value: 0.7207747803979043 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.7933084651006226, 'n_estimators': 438, 'learning_rate': 0.0796244585080611}. Best is trial 19 with value: 0.7378573705369926.
Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.8382352941176471
Fold

Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.6741071428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 16:00:11,829] Trial 38 finished with value: 0.7207747803979043 and parameters: {'subsample': 0.23911041397514493, 'dropout_rate': 0.47419599199828566, 'n_estimators': 249, 'learning_rate': 0.06698521350410276}. Best is trial 28 with value: 0.74162050985486.
Fold 1 C-index: 0.658008658008658
Fold 2 C-index: 0.6919642857142857
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.6126760563380281
[I 2024-04-14 16:00:13,579] Trial 39 finished with value: 0.7078251586619834 and parameters: {'subsample': 0.4245656147243987, 'dropout_rate': 0.7665133891733147, 'n_estimators': 224, 'learning_rate': 0.05575307634287385}. Best is trial 28 with value: 0.74162050985486.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index

...

Fold 3 IBS: 0.1783999568257794
Fold 4 IBS: 0.20183880070145502
Fold 5 IBS: 0.2121882393181561
[I 2024-04-14 16:04:27,362] Trial 19 finished with value: 0.19835046284707328 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.19835046284707328.
Fold 1 IBS: 0.1922099266697023
Fold 2 IBS: 0.20500626401911293
Fold 3 IBS: 0.1819421294378562
Fold 4 IBS: 0.20531861803436055
Fold 5 IBS: 0.21298081915188694
[I 2024-04-14 16:04:27,702] Trial 20 finished with value: 0.1994915514625838 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.19835046284707328.
Fold 1 IBS: 0.20013778642266292
Fold 2 IBS: 0.20393569161683953
Fold 3 IBS: 0.18712698518091442
Fold 4 IBS: 0.20944879242499856
Fold 5 IBS: 0.21303933620208118
[I 2024-04-14 16:04:28,017] Trial 21 finis

Fold 3 IBS: 0.1685675012002108
Fold 4 IBS: 0.20909390394783278
Fold 5 IBS: 0.23159412949218366
[I 2024-04-14 16:04:45,106] Trial 38 finished with value: 0.20072232473067647 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.13272164755980653, 'n_estimators': 197, 'learning_rate': 0.034653901878161156}. Best is trial 33 with value: 0.19303761371008804.
Fold 1 IBS: 0.1763284024432531
Fold 2 IBS: 0.24237849570630937
Fold 3 IBS: 0.16838621148605842
Fold 4 IBS: 0.23433215637540045
Fold 5 IBS: 0.23218136784009083
[I 2024-04-14 16:04:46,013] Trial 39 finished with value: 0.21072132677022243 and parameters: {'subsample': 0.167666800643059, 'dropout_rate': 0.9404469958471148, 'n_estimators': 122, 'learning_rate': 0.05575307634287385}. Best is trial 33 with value: 0.19303761371008804.
Fold 1 IBS: 0.1835607046663353
Fold 2 IBS: 0.20550258426704343
Fold 3 IBS: 0.17033281078924473
Fold 4 IBS: 0.20234480009040523
Fold 5 IBS: 0.21318875015485583
[I 2024-04-14 16:04:46,504] Trial 40 

Fold 3 IBS: 0.16822611367610577
Fold 4 IBS: 0.2337714954670114
Fold 5 IBS: 0.23499642019553307
[I 2024-04-14 16:05:06,966] Trial 57 finished with value: 0.2120023287068955 and parameters: {'subsample': 0.13688965562363797, 'dropout_rate': 0.9526896712757614, 'n_estimators': 130, 'learning_rate': 0.055651931121615976}. Best is trial 33 with value: 0.19303761371008804.
Fold 1 IBS: 0.20180718166088418
Fold 2 IBS: 0.20823056304136786
Fold 3 IBS: 0.19148854447397767
Fold 4 IBS: 0.2174086350391278
Fold 5 IBS: 0.21280121650803058
[I 2024-04-14 16:05:07,315] Trial 58 finished with value: 0.2063472281446776 and parameters: {'subsample': 0.7132233145884603, 'dropout_rate': 0.7590040418023142, 'n_estimators': 30, 'learning_rate': 0.03238483343413252}. Best is trial 33 with value: 0.19303761371008804.
Fold 1 IBS: 0.18523772398626553
Fold 2 IBS: 0.20192659760289666
Fold 3 IBS: 0.17581399422575117
Fold 4 IBS: 0.2042588924101933
Fold 5 IBS: 0.21198594062334492
[I 2024-04-14 16:05:07,965] Trial 59 fin

Fold 4 IBS: 0.19977802798795471
Fold 5 IBS: 0.21147728525732398
[I 2024-04-14 16:05:21,029] Trial 76 finished with value: 0.1954331545259906 and parameters: {'subsample': 0.2401639575567464, 'dropout_rate': 0.7857326186782545, 'n_estimators': 28, 'learning_rate': 0.08898292294449694}. Best is trial 75 with value: 0.19240580355897693.
Fold 1 IBS: 0.17686859585016973
Fold 2 IBS: 0.21612375928723943
Fold 3 IBS: 0.16351449690372113
Fold 4 IBS: 0.20921833604228476
Fold 5 IBS: 0.21934697817438564
[I 2024-04-14 16:05:21,509] Trial 77 finished with value: 0.19701443325156015 and parameters: {'subsample': 0.20835315159322065, 'dropout_rate': 0.6981056962456518, 'n_estimators': 55, 'learning_rate': 0.08671273256255693}. Best is trial 75 with value: 0.19240580355897693.
Fold 1 IBS: 0.1790517521200231
Fold 2 IBS: 0.20778503734954273
Fold 3 IBS: 0.1687165120258161
Fold 4 IBS: 0.1864366058034224
Fold 5 IBS: 0.21459658176413984
[I 2024-04-14 16:05:21,863] Trial 78 finished with value: 0.1913172978125

Fold 5 IBS: 0.21623431750226857
[I 2024-04-14 16:05:41,158] Trial 95 finished with value: 0.21235001060435743 and parameters: {'subsample': 0.18849122365713838, 'dropout_rate': 0.8183000737870284, 'n_estimators': 63, 'learning_rate': 0.0058046127253871885}. Best is trial 78 with value: 0.19131729781258883.
Fold 1 IBS: 0.17920744733080823
Fold 2 IBS: 0.20650083821997772
Fold 3 IBS: 0.16787219115746788
Fold 4 IBS: 0.1935736735660491
Fold 5 IBS: 0.2172044375016859
[I 2024-04-14 16:05:41,627] Trial 96 finished with value: 0.19287171755519777 and parameters: {'subsample': 0.10202382959971797, 'dropout_rate': 0.7861202496553675, 'n_estimators': 78, 'learning_rate': 0.05227568594858167}. Best is trial 78 with value: 0.19131729781258883.
Fold 1 IBS: 0.1811475437618194
Fold 2 IBS: 0.2024565347988353
Fold 3 IBS: 0.17222008300034344
Fold 4 IBS: 0.1956371349303774
Fold 5 IBS: 0.21757937566078778
[I 2024-04-14 16:05:42,259] Trial 97 finished with value: 0.19380813443043265 and parameters: {'subsamp

In [182]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [183]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.742
train_ibs:  0.191


#### Test

In [184]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [185]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.9261509226223655,
                                              learning_rate=0.04202399206134448,
                                              n_estimators=274,
                                              random_state=123,
                                              subsample=0.10061641287468016)

C-index score: 0.667


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.8516991920191342,
                                              learning_rate=0.09808413258788978,
                                              n_estimators=36, random_state=123,
                                              subsample=0.17910269140503615)

IBS: 0.205


In [186]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [187]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.798,1.0
Randomsurvivalforest,0.787,2.0
GradientBoosting,0.743,3.0
ComponentwiseGradientBoosting,0.742,4.0
CoxPH,0.737,5.0
CoxLasso,0.736,6.5
CoxElastic,0.736,6.5
CoxRidge,0.647,8.0


In [188]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
ExtraSurvivalTrees,0.184,1.0
Randomsurvivalforest,0.185,2.0
CoxPH,0.187,4.0
CoxLasso,0.187,4.0
CoxElastic,0.187,4.0
ComponentwiseGradientBoosting,0.191,6.0
GradientBoosting,0.214,7.0
CoxRidge,0.217,8.0


In [189]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.708,1.0
CoxRidge,0.701,2.0
ComponentwiseGradientBoosting,0.667,3.0
ExtraSurvivalTrees,0.632,4.0
CoxElastic,0.598,5.0
CoxLasso,0.597,6.0
CoxPH,0.593,7.0
GradientBoosting,0.590,8.0


In [190]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ComponentwiseGradientBoosting,0.205,1.0
Randomsurvivalforest,0.208,2.0
ExtraSurvivalTrees,0.217,3.0
GradientBoosting,0.218,4.0
CoxRidge,0.221,5.0
CoxPH,0.269,6.0
CoxLasso,0.272,7.5
CoxElastic,0.272,7.5


In [191]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/yeojohnson/plsr/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_os_yeojohnson_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [192]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
